In [2]:
!pip install langchain

In [3]:
!pip install -U langchain-google-genai

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 68.8/68.8 kB 3.2 MB/s eta 0:00:00


In [4]:
!pip install langchain-tavily

In [10]:
import requests
from google.colab import userdata
from langchain.tools import tool
from langchain_tavily import TavilySearch
from langchain.chat_models import init_chat_model
from langchain.agents import create_agent
from langgraph.checkpoint.memory import InMemorySaver

TAVILY_API_KEY = userdata.get("TAVILY_API_KEY")
RAPIDAPI_KEY = userdata.get("RAPIDAPI_KEY")
GOOGLE_API_KEY = userdata.get("GOOGLE_API_KEY")

skill_demand_tool = TavilySearch(
    max_results=5,
    search_depth="advanced",
    tavily_api_key=TAVILY_API_KEY,
)

@tool
def search_jobs(skill: str, location: str) -> list:
    """
    Search for jobs requiring a specific skill using the JSearch API.
    """
    print("\nCalling search_jobs tool")
    print(f"Searching jobs for: {skill} in {location}")
    url = "https://jsearch.p.rapidapi.com/search"
    headers = {
        "x-rapidapi-key": RAPIDAPI_KEY,
        "x-rapidapi-host": "jsearch.p.rapidapi.com",
    }
    params = {
        "query": f"{skill} in {location}",
        "page": "1",
        "num_pages": "1",
        "country": "in",
        "employment_types": "INTERN,FULLTIME",
        "job_requirements": "no_experience,under_3_years_experience",
    }
    response = requests.get(url, headers=headers, params=params)
    data = response.json()
    jobs = data.get("data", [])
    print(f"Found {len(jobs)} jobs\n")
    return [
        {
            "title": job.get("job_title"),
            "company": job.get("employer_name"),
            "location": job.get("job_city"),
            "apply_link": job.get("job_apply_link"),
        }
        for job in jobs
    ]

SYSTEM_PROMPT = """
You are a Skill-to-Career Mapping assistant that helps students understand skill demand
and find matching job opportunities.

You have access to these tools:
- skill_demand_tool: Research industry demand, salary insights, and career trends
- search_jobs: Find real job listings based on skills and location

Present results in a clean, readable format with clear sections and spacing.
Include all job details with apply links.
Do not use markdown formatting.
"""

model = init_chat_model(
    "google_genai:gemini-2.5-flash",
    api_key=GOOGLE_API_KEY,
)

checkpointer = InMemorySaver()
config = {"configurable": {"thread_id": "1"}}

agent = create_agent(
    model=model,
    system_prompt=SYSTEM_PROMPT,
    tools=[skill_demand_tool, search_jobs],
    checkpointer=checkpointer,
    debug=True,
)

user_query = (
    "What's the demand for generative AI in the industry "
    "and show me related job openings in India"
)

response = agent.invoke({
  "messages": [{"role": "user", "content": user_query}]
}, config=config)

print(response["messages"][-1].content[0]["text"])

user_query = "Tell me more about the second job you showed"

response = agent.invoke(
    {"messages": [{"role": "user", "content": user_query}]},
    config=config,
)


[values] {'messages': [HumanMessage(content="What's the demand for generative AI in the industry and show me related job openings in India", additional_kwargs={}, response_metadata={}, id='a119714f-edbc-40fa-9c72-f49eded58345')]}
[updates] {'model': {'messages': [AIMessage(content='', additional_kwargs={'function_call': {'name': 'search_jobs', 'arguments': '{"location": "India", "skill": "generative AI"}'}, '__gemini_function_call_thought_signatures__': {'d8f3e4a0-8dc6-4e94-b47c-66dac8cb676f': 'Ct8GAQw51sfv3NBcVwj9WheN7YQ/7sWlT4rj14RF+8py4l8hWZLAVZ+nunKnbQyJSU9rubqTGcn2uM4qstkXr7OBJFgg/sdyR6N9MO7XmegaA/DU19Si8Tv55IfK8YkiHWaQNabLjwmQzOMoBDDMUxsOg67XVCXJEbYkxw+AALbF6G4I26dFCnGplPkq3TiQg9DUCpNEg47YO2a4n/ibtrVbj4eCfBj7hjc4jevzsafH/BFuyYkQbLHPkNTEzLbXWroIDe4bcrY/8C4wPwRtB1o6qYqtX/+jc5CkSHL/O5bWAoTaYXZTlCsNqu3I2KhaBKp+liIyV3m178UIsaWZKGf7e/knbx7VaALSOPRXhnpG21kQ1jIVZUyzxnxo5MTSZph9v335aC1Eun565w168dYYm5hqXuuXdLXEkx97svHfAhyCTB/uL8eVVHYnvY2xl0MkZ9IcVBveZIb9ZhHIXN6yeOjIM2tDDWtfMIBktmeGpCeBop/8

In [11]:
print(response["messages"][-1].content[0]["text"])

I can only provide the details that were retrieved for the job openings. For the "Senior Project Manager - Generative AI Data Operations" position at Welocalize in Parashurampuri, the available information is:

  Title: Senior Project Manager - Generative AI Data Operations
  Company: Welocalize
  Location: Parashurampuri
  Apply Link: https://www.learn4good.com/jobs/gurugram/india/info_technology/5139491465/e/

I do not have access to further details such as a full job description or specific requirements beyond what is listed. You may find more information by visiting the apply link directly.
